In [7]:
import tensorrt as trt
import os.path

In [ ]:
logger = trt.Logger(trt.Logger.WARNING)
explicit_batch = 1 << (int)(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)  # trt7
batch_size = 4
model = "../Models/beachbot_yolov5s_beach-cleaning-object-detection__v8-yolotrain__yolov5pytorch_1280_finetune/best.onnx"
output = model.rsplit(".",1)[0] + ".engine"


if os.path.isfile(output):
    print(f"Output file {output} exists, skipping model conversion!")
else:
    print("Create tensorrt file", output)

    with trt.Builder(logger) as builder:
        with builder.create_network(explicit_batch) as network:
            with trt.OnnxParser(network, logger) as parser:
                with builder.create_builder_config() as builder_config:
                    #builder.fp16_mode = True # optional
                    #builder_config.max_workspace_size = workspace_size * (1024 * 1024)
                    builder_config.set_flag(trt.BuilderFlag.FP16)
                    #builder_config.set_flag(trt.BuilderFlag.INT8)
                    with open(model, 'rb') as f:
                        print('Beginning ONNX file parsing')
                        if not parser.parse(f.read()):
                            for error in range(parser.num_errors):
                                print("ERROR", parser.get_error(error))
                    print("num layers:", network.num_layers)
                    print("out shape:", network.get_input(0).shape)
                    #network.get_input(0).shape = [batch_size, 3, 608, 608]  # trt7
                    engine = builder.build_serialized_network(network, builder_config)
                    if engine is not None:
                        #engine = builder.build_cuda_engine(network)
                        with open(output, 'wb') as f:
                            f.write(engine.serialize())
                        print("Completed creating Engine")
                    else:
                        print(f"Error, model conversion failed, engin is none.")




Create tensorrt file ../Models/beachbot_yolov5s_beach-cleaning-object-detection__v8-yolotrain__yolov5pytorch_1280_finetune/best.engine
Beginning ONNX file parsing
num layers: 264
out shape: (1, 3, 800, 1280)


AttributeError: 'tensorrt_bindings.tensorrt.Builder' object has no attribute 'build_cuda_engine'

In [ ]:
builder.f